In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift/')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from text_helpers.CustomBertModel import CustomBertForSequenceClassification

import torch
import numpy as np
from sklearn.metrics import accuracy_score
from math import ceil

device = 'cuda'
batch_size = 256

In [5]:
checkpoint_name = "fabriceyhc/bert-base-uncased-dbpedia_14"

tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

model_original = AutoModelForSequenceClassification.from_pretrained(checkpoint_name, num_labels=14)
model_original = model_original.eval().to(device)

model_relu = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=14)
model_relu = model_relu.eval().to(device)

dataset = load_dataset("fancyzhx/dbpedia_14")
test_texts = list(dataset["test"]["content"])
test_labels = np.array(dataset["test"]["label"])

preds_original = []
preds_relu = []
n_batches = ceil(len(test_texts) / batch_size)

with torch.no_grad():
    for i in range(n_batches):
        batch = test_texts[i*batch_size:(i+1)*batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits_original = model_original(input_ids=input_ids, attention_mask=attention_mask).logits
        logits_relu = model_relu(input_ids, attention_mask)

        preds_original.extend(logits_original.argmax(dim=-1).cpu().numpy())
        preds_relu.extend(logits_relu.argmax(dim=-1).cpu().numpy())

        if (i+1) % 20 == 0:
            print(f"DBpedia batch {i+1}/{n_batches}")

acc_dbpedia_original = accuracy_score(test_labels, preds_original)
acc_dbpedia_relu = accuracy_score(test_labels, preds_relu)

print(f"DBpedia-14 ({len(test_texts)} test samples)")
print(f"  original (tanh):    {acc_dbpedia_original:.4f}")
print(f"  substituted (ReLU): {acc_dbpedia_relu:.4f}")

config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

dbpedia_14/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.3MB            

dbpedia_14/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

DBpedia batch 20/274
DBpedia batch 40/274
DBpedia batch 60/274
DBpedia batch 80/274
DBpedia batch 100/274
DBpedia batch 120/274
DBpedia batch 140/274
DBpedia batch 160/274
DBpedia batch 180/274
DBpedia batch 200/274
DBpedia batch 220/274
DBpedia batch 240/274
DBpedia batch 260/274
DBpedia-14 (70000 test samples)
  original (tanh):    0.9903
  substituted (ReLU): 0.9902


In [6]:
del model_original, model_relu
torch.cuda.empty_cache()

In [7]:
checkpoint_name = "rttl-ai/bert-base-uncased-yelp-reviews"

tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

model_original = AutoModelForSequenceClassification.from_pretrained(checkpoint_name, num_labels=5)
model_original = model_original.eval().to(device)

model_relu = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=5)
model_relu = model_relu.eval().to(device)

dataset = load_dataset("Yelp/yelp_review_full")
test_texts = list(dataset["test"]["text"])
test_labels = np.array(dataset["test"]["label"])

preds_original = []
preds_relu = []
n_batches = ceil(len(test_texts) / batch_size)

with torch.no_grad():
    for i in range(n_batches):
        batch = test_texts[i*batch_size:(i+1)*batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits_original = model_original(input_ids=input_ids, attention_mask=attention_mask).logits
        logits_relu = model_relu(input_ids, attention_mask)

        preds_original.extend(logits_original.argmax(dim=-1).cpu().numpy())
        preds_relu.extend(logits_relu.argmax(dim=-1).cpu().numpy())

        if (i+1) % 20 == 0:
            print(f"Yelp batch {i+1}/{n_batches}")

acc_yelp_original = accuracy_score(test_labels, preds_original)
acc_yelp_relu = accuracy_score(test_labels, preds_relu)

print(f"Yelp Review Full ({len(test_texts)} test samples)")
print(f"  original (tanh):    {acc_yelp_original:.4f}")
print(f"  substituted (ReLU): {acc_yelp_relu:.4f}")

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

...

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Yelp batch 20/196
Yelp batch 40/196
Yelp batch 60/196
Yelp batch 80/196
Yelp batch 100/196
Yelp batch 120/196
Yelp batch 140/196
Yelp batch 160/196
Yelp batch 180/196
Yelp Review Full (50000 test samples)
  original (tanh):    0.7132
  substituted (ReLU): 0.7088


In [8]:
del model_original, model_relu
torch.cuda.empty_cache()

In [ ]:
checkpoint_name = "fabriceyhc/bert-base-uncased-yahoo_answers_topics"

tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

model_original = AutoModelForSequenceClassification.from_pretrained(checkpoint_name, num_labels=10)
model_original = model_original.eval().to(device)

model_relu = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=10)
model_relu = model_relu.eval().to(device)

dataset = load_dataset("community-datasets/yahoo_answers_topics")
test_texts = [" ".join(x for x in (t, q, a) if x)
              for t, q, a in zip(dataset["test"]["question_title"], dataset["test"]["question_content"], dataset["test"]["best_answer"])]
test_labels = np.array(dataset["test"]["topic"])

preds_original = []
preds_relu = []
n_batches = ceil(len(test_texts) / batch_size)

with torch.no_grad():
    for i in range(n_batches):
        batch = test_texts[i*batch_size:(i+1)*batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits_original = model_original(input_ids=input_ids, attention_mask=attention_mask).logits
        logits_relu = model_relu(input_ids, attention_mask)

        preds_original.extend(logits_original.argmax(dim=-1).cpu().numpy())
        preds_relu.extend(logits_relu.argmax(dim=-1).cpu().numpy())

        if (i+1) % 20 == 0:
            print(f"Yahoo batch {i+1}/{n_batches}")

acc_yahoo_original = accuracy_score(test_labels, preds_original)
acc_yahoo_relu = accuracy_score(test_labels, preds_relu)

print(f"Yahoo Answers Topics ({len(test_texts)} test samples)")
print(f"  original (tanh):    {acc_yahoo_original:.4f}")
print(f"  substituted (ReLU): {acc_yahoo_relu:.4f}")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

yahoo_answers_topics/test-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 21.9MB            

yahoo_answers_topics/test-00000-of-00001(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Yahoo batch 20/235
Yahoo batch 40/235
Yahoo batch 60/235
Yahoo batch 80/235
Yahoo batch 100/235
Yahoo batch 120/235
Yahoo batch 140/235
Yahoo batch 160/235
Yahoo batch 180/235
Yahoo batch 200/235
Yahoo batch 220/235
Yahoo Answers Topics (60000 test samples)
  original (tanh):    0.7499
  substituted (ReLU): 0.7505


In [10]:
del model_original, model_relu
torch.cuda.empty_cache()

In [11]:
checkpoint_name = "fabriceyhc/bert-base-uncased-ag_news"

tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

model_original = AutoModelForSequenceClassification.from_pretrained(checkpoint_name, num_labels=4)
model_original = model_original.eval().to(device)

model_relu = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=4)
model_relu = model_relu.eval().to(device)

dataset = load_dataset("fancyzhx/ag_news")
test_texts = list(dataset["test"]["text"])
test_labels = np.array(dataset["test"]["label"])

preds_original = []
preds_relu = []
n_batches = ceil(len(test_texts) / batch_size)

with torch.no_grad():
    for i in range(n_batches):
        batch = test_texts[i*batch_size:(i+1)*batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits_original = model_original(input_ids=input_ids, attention_mask=attention_mask).logits
        logits_relu = model_relu(input_ids, attention_mask)

        preds_original.extend(logits_original.argmax(dim=-1).cpu().numpy())
        preds_relu.extend(logits_relu.argmax(dim=-1).cpu().numpy())

        if (i+1) % 20 == 0:
            print(f"AGNews batch {i+1}/{n_batches}")

acc_agnews_original = accuracy_score(test_labels, preds_original)
acc_agnews_relu = accuracy_score(test_labels, preds_relu)

print(f"AG News ({len(test_texts)} test samples)")
print(f"  original (tanh):    {acc_agnews_original:.4f}")
print(f"  substituted (ReLU): {acc_agnews_relu:.4f}")

config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

AGNews batch 20/30
AG News (7600 test samples)
  original (tanh):    0.9375
  substituted (ReLU): 0.9379


In [ ]:
import csv

with open('/content/drive/MyDrive/results/relu_substitution_accuracy.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['dataset', 'n_test_samples', 'original_tanh', 'substituted_relu', 'difference'])
    writer.writerow(['DBpedia-14', 70000, acc_dbpedia_original, acc_dbpedia_relu,
                     acc_dbpedia_original - acc_dbpedia_relu])
    writer.writerow(['Yelp Review Full', 50000, acc_yelp_original, acc_yelp_relu,
                     acc_yelp_original - acc_yelp_relu])
    writer.writerow(['Yahoo Answers Topics', 60000, acc_yahoo_original, acc_yahoo_relu,
                     acc_yahoo_original - acc_yahoo_relu])
    writer.writerow(['AG News', 7600, acc_agnews_original, acc_agnews_relu,
                     acc_agnews_original - acc_agnews_relu])